<a href="https://colab.research.google.com/github/cbonnin88/WattWise/blob/main/WattWise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit
!pip install weasyprint
!pip install pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 41.6 MB/s eta 0:00:00
  Attempting uninstall: tinycss2
    Found existing installation: tinycss2 1.4.0
    Uninstalling tinycss2-1.4.0:
      Successfully uninstalled tinycss2-1.4.0


In [24]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from weasyprint import HTML
import base64
from wordcloud import WordCloud



# --- PAGE CONFIG ---
# Sets the dashboard title and wide layout for better data visualizations
st.set_page_config(page_title='WattWise',layout='wide')


# --- SIDEBAR: PM CONTROLs ---
# These parameters allow our Product Managers to perform sensitivity analysis
st.sidebar.header('Test Configuration')
lift_input = st.sidebar.slider('Simulated Treatment Life (%)',0,20,8) / 100
sample_size = st.sidebar.number_input('Total User Sample',500,5000,1500)
z_threshold = st.sidebar.slider('Outlier Sensitivity (Z-Score)',1.0,4.0,3.0)

st.sidebar.header('Revenue Assumptions')
energy_price = st.sidebar.slider('Energy Price (€/kWh)',0.20,0.60,0.40)
kwh_per_hour = 7.4 # Average Level 2 EV charging speed
total_market_size = st.sidebar.number_input('Projected Total Users',value=10000)


# --- 0.5 GRID CONNECTION STATUS INDICATOR ---
# Simulates a live heartbeat from Western European energy grids
status_cols = st.columns(4)
countries = ['Germany', 'France', 'United Kingdom', 'Netherlands']

for i, country in enumerate(countries):
    with status_cols[i]:
        # Using a green circle emoji to represent a healthy connection
        st.caption(f"🟢 {country} Grid: **Connected**")

st.divider() # Adds a clean visual break before the metrics


# --- 1. DATA GENERATION & CLEANING ---
@st.cache_data
def generated_cleansed_data(n,lift):
  'Generating synthetic charging data for Western European markets'
  np.random.seed(42)
  # Simulate normal charging hours (Mean=6, StdDev=1)
  hours = np.random.normal(6,1.5,n)
  # Inject anomalies/outliers to test the Z-Score filter
  hours[0:3] = [24.0,-5.0,30.0]

  df_product = pd.DataFrame({
      'group': np.random.choice(['control','treatment'],n),
      'engagement_hours': hours,
      'country': np.random.choice(['Germany','France','United Kingdom','Netherlands'],n)
  })

  # Conversion Logic: Lift applied to treatment group
  df_product['conv_prob'] =0.12
  df_product.loc[df_product['group'] == 'treatment','conv_prob'] += lift
  df_product['converted'] = np.random.binomial(1, df_product['conv_prob'].clip(0,1))

  return df_product.drop(columns=['conv_prob'])

df_raw = generated_cleansed_data(sample_size, lift_input)

# Outlier Fitler using Z-Score method for Normal Distribution
mean_val = df_raw['engagement_hours'].mean()
std_val = df_raw['engagement_hours'].std()
z_scores = np.abs((df_raw['engagement_hours'] - mean_val) / std_val)
df_clean = df_raw[z_scores <= z_threshold].copy()
outliers_found = len(df_raw) - len(df_clean)

# Financial Calculation
df_clean['revenue'] = df_clean['engagement_hours'] * kwh_per_hour * energy_price


# --- 2. STATISTICAL ANALYSIS ---
# Performs a two-proportion Z-test to determine Success Probability
conversions = df_clean.groupby('group')['converted'].sum()
totals = df_clean.groupby('group')['converted'].count()
count = [conversions.get('treatment',0),conversions.get('control',0)]
nobs = [totals.get('treatment',1), totals.get('control',1)]
z_stat, p_val = proportions_ztest(count,nobs)
success_prob = (1 - p_val) * 100

# --- 3. DASHBOARD UI ---
st.title('🌱 WattWise Smart-Charging Analysis')
st.markdown(f'**Market:** Western Europe | **Cleaned Data:** {len(df_clean)} users ({outliers_found}) outliers removed')

# Metrics Cards
c1,c2,c3,c4 = st.columns(4)
cr_ctrl = df_clean[df_clean['group'] == 'control']['converted'].mean()
cr_treat = df_clean[df_clean['group'] == 'treatment']['converted'].mean()

c1.metric('Control CR',f'{cr_ctrl: .1%}')
c2.metric('Treatment CR',f'{cr_treat: .1%}')
c3.metric('Success Prob.',f'{success_prob:.1f}%')
c4.metric('P-Value',f'{p_val:.4f}')

# Financial Impact Section
rev_ctrl = df_clean[df_clean['group'] == 'control']['revenue'].mean()
rev_treat = df_clean[df_clean['group'] == 'treatment']['revenue'].mean()
rev_lift_per_user = rev_treat - rev_ctrl
annual_impact = rev_lift_per_user * total_market_size * 52

st.subheader('💰 Business Impact')
rm1,rm2,rm3 = st.columns(3)
rm1.metric('Avg Rev (Control)',f'€{rev_ctrl:.2f}')
rm2.metric('Avg Rev (Treatment)',f'€{rev_treat:.2f}')
rm3.metric('Projected Annual Lift',f'€{annual_impact:.0f}')


# --- 3.5 GEOGRAPHIC PERFORMANCE CHART ---
st.subheader("📊 Conversion Rate by Country")

# 1. Aggregate data for the chart
# We calculate the mean of 'converted' which gives us the conversion rate per segment
country_stats = df_clean.groupby(['country', 'group'])['converted'].mean().reset_index()
country_stats['Conversion Rate (%)'] = country_stats['converted'] * 100

# 2. Create the Plotly Bar Chart
fig_country = px.bar(
    country_stats,
    x='country',
    y='Conversion Rate (%)',
    color='group',
    barmode='group', # Puts bars side-by-side for easy comparison
    text_auto='.1f', # Shows the percentage value on top of the bars[cite: 2]
    title="Comparison of AI vs. Manual Charging by Market",
    labels={'country': 'Western European Market', 'group': 'Test Group'},
    color_discrete_map={'control': '#636EFA', 'treatment': '#00CC96'} # GreenTech branding[cite: 2]
)

# 3. Fine-tune layout for clarity[cite: 2]
fig_country.update_layout(yaxis_ticksuffix="%", margin=dict(l=20, r=20, b=20, t=50))
st.plotly_chart(fig_country, use_container_width=True)


# --- 4. VISUALIZATION ---
st.subheader('Engagement Distribution (Post-Cleaning)')
fig = px.histogram(
    df_clean,
    x='engagement_hours',
    color='group',
    barmode='overlay',
    title = 'User Charging Hours Distribution',
    color_discrete_map = {'control':'#636EFA','treatment':'#00CC96'},
    labels = {'engagement_hours':'Engagement (Hours)','count':'Users'}
)
st.plotly_chart(fig,use_container_width=True)

# --- 4.5 QUALITATIVE ANALYSIS: WORD CLOUD ---
st.divider()
st.subheader('💬 User Sentiment Analysis')

treatment_feedback = [
    "The smart charging saved me 20 Euro this week!", "Love the eco-friendly UI",
    "Seamless integration with my home charger", "Innovative AI features",
    "Automatic scheduling is a game changer", "Saving money on off-peak hours",
    "Great for the planet", "Very efficient and sustainable", "The app is so smart",
    "Best GreenTech app in Germany", "Highly recommend the automated charging",
    "Carbon footprint tracking is helpful", "Zero hassle energy management"
]

control_feedback = [
    "It's just a basic charging app", "Manual scheduling is tedious",
    "Pricey during peak hours", "Standard features", "Confusing UI at times",
    "It's okay but nothing special", "Wish it was more automated",
    "Traditional charging interface", "Expensive in the UK",
    "Functional but slow", "Average experience", "Old fashioned layout"
]

# 2. UI Toggle
sentiment_group = st.radio("Analyze Feedback for:", ["Treatment", "Control"], horizontal=True)

if sentiment_group == 'Treatment':
  cloud_text = ' '.join(np.random.choice(treatment_feedback,200))
else:
  cloud_text = ' '.join(np.random.choice(control_feedback,200))

# 3. Create and Display WordCloud
wc = WordCloud(
    width = 1000,
    height = 500,
    background_color = 'white',
    colormap='viridis_r'
).generate(cloud_text)

wc_array = np.array(wc)
fig_wc = px.imshow(wc_array)

# Hide axes for a clean visualization
fig_wc.update_xaxes(showticklabels=False)
fig_wc.update_yaxes(showticklabels=False)
fig_wc.update_layout(margin=dict(l=10, r=10, b=10, t=10))

st.plotly_chart(fig_wc, use_container_width=True)

st.subheader("Recent Comments")
sample_comments = np.random.choice(treatment_feedback if sentiment_group == "Treatment" else control_feedback, 3)
for comment in sample_comments:
    st.info(f"“{comment}”")

st.divider()
st.header("🌍 Regional User Sentiment")

# 1. Define a more granular feedback database
regional_feedback = {
    'Germany': {
        'treatment': ["Efficiency is top-notch", "Saves me Euro on every charge", "Very reliable AI"],
        'control': ["Standard charging speed", "UI could be better", "A bit expensive"]
    },
    'France': {
        'treatment': ["Superbe application", "Smart charging is seamless", "Eco-friendly approach"],
        'control': ["Basic features", "Manual work required", "Average experience"]
    },
    'Netherlands': {
        'treatment': ["Love the smart-grid integration", "Innovative and green", "Very easy to use"],
        'control': ["Missing advanced features", "Okay for daily use", "Pricey during peaks"]
    },
    'United Kingdom': {
        'treatment': ["Saving plenty on energy bills", "Brilliant smart-charging", "Great for my EV"],
        'control': ["Manual scheduling is a faff", "Too expensive", "Standard layout"]
    }
}

# 2. UI for selection
col_f1, col_f2 = st.columns(2)
with col_f1:
    f_country = st.selectbox("Select Country for Feedback:", list(regional_feedback.keys()))
with col_f2:
    f_group = st.radio("Group:", ["Treatment", "Control"], horizontal=True).lower()

# 3. Filter and generate text
# We fetch the specific list and expand it to create a dense cloud
specific_comments = regional_feedback[f_country][f_group]
cloud_text = " ".join(np.random.choice(specific_comments, 200))

# 4. Generate and Plot with Plotly
wc = WordCloud(width=1000, height=400, background_color='white', colormap='YlGn').generate(cloud_text)
fig_wc = px.imshow(np.array(wc))
fig_wc.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig_wc.update_layout(margin=dict(l=10, r=10, b=10, t=10))

st.plotly_chart(fig_wc, use_container_width=True)

# 5. Show localized raw comments
st.subheader(f"Raw Feedback from {f_country}")
for comment in np.random.choice(specific_comments, 2):
    st.info(f"“{comment}”")




# --- 5. EXPORT SECTION ---
st.divider()
st.subheader('📁 Export Results')
col_dl1, col_dl2 = st.columns(2)

# CSV Export: Direct Download of cleaned session data
csv = df_clean.to_csv(index=False).encode('utf-8')
col_dl1.download_button(
    label='Download Raw Cleaned Data (CSV)',
    data=csv,
    file_name='wattwise_ab_test_cleaned.csv',
    mime='text/csv'
)


# PDF Export: Professional HTML-to-PDF Executive Summary
product_report = f"""
<html>
<head>
    <style>
        body {{ font-family: Arial, sans-serif; color: #2D3748; padding: 40px; }}
        .header {{ background: #2F855A; color: white; padding: 20px; border-radius: 5px; }}
        .metric-box {{ border: 1px solid #E2E8F0; padding: 15px; margin-top: 20px; border-radius: 5px; }}
        h2 {{ color: #2F855A; }}
    </style>
</head>
<body>
    <div class="header"><h1>WattWise A/B Test Report</h1></div>
    <div class="metric-box">
        <h2>Executive Results</h2>
        <p><b>Success Probability:</b> {success_prob:.2f}%</p>
        <p><b>Statistically Significant:</b> {"Yes" if p_val < 0.05 else "No"}</p>
        <p><b>Projected Annual Revenue Lift:</b> €{annual_impact:,.2f}</p>
    </div>
    <p>This report summarizes the performance of the Smart Charging feature across {sample_size} users in Western Europe</p>
</body>
</html>
"""
pdf_data = HTML(string=product_report).write_pdf()
col_dl2.download_button(
    label = 'Download Executive Summary (PDF)',
    data=pdf_data,
    file_name='WattWise_product_report.pdf',
    mime='application/pdf'
)

Overwriting app.py


In [25]:
from pyngrok import ngrok
import os
from google.colab import userdata

ngrok.kill()

ngrok_auth_token = userdata.get('ngrok_token')
ngrok.set_auth_token(ngrok_auth_token)

public_url = ngrok.connect(8501,proto='http')
print(f'WattWise Dashboard is live at: {public_url}')

!streamlit run app.py --server.port 8501 &

WattWise Dashboard is live at: NgrokTunnel: "https://8bae-34-75-243-154.ngrok-free.app" -> "http://localhost:8501"


2026-05-11 20:24:59.012 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.75.243.154:8501

2026-05-11 20:25:09.965 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-05-11 20:25:10.038 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-05-11 20:25:11.540 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=